In [1]:
import math
import numpy as np

### Ejercicio 7.
Considerar un sistema de colas con un único servidor que recibe solicitudes de acuerdo con un proceso de Poisson no homogéneo, cuya tasa es inicialmente de $4$ solicitudes por hora, aumenta linealmente hasta alcanzar $19$ solicitudes por hora luego de $5$ horas, y posteriormente disminuye linealmente hasta volver a $4$ solicitudes por hora luego de otras $5$ horas. Este comportamiento de la tasa se repite indefinidamente; es decir,
$$ \lambda(t + 10) = \lambda(t) $$

Suponer que:
- El tiempo de servicio del servidor se distribuye de manera exponencial, con una tasa de $25$ servicios por hora.
- Siempre que el servidor completa un trabajo y no encuentra trabajos para realizar, toma un descanso por un tiempo uniformemente distribuido en el intervalo $(0, 0.3)$ horas.
- Si al retomar no encuentra trabajos para realizar, vuelve a tomarse un descanso con la misma distribución.

Se pide:

a. Desarrollar un programa que simule el proceso durante un tiempo $T = 100$ horas, registrando los tiempos de llegada, los tiempos de servicio, los períodos de descanso del servidor y la evolución del número de trabajos en cola.

b. Utilizar el programa desarrollado en $a)$ para estimar el tiempo esperado total que el servidor permanece en descanso durante las primeras $100$ horas de operación. Detener las simulaciones cuando la desviación estándar muestral del estimador de la media sea menor que $0.05$ horas.

c. Estimar número esperado de trabajos que el servidor finaliza luego del tiempo $T$ . Detener las simulaciones cuando la desviación estándar muestral del estimador de la media sea menor que $0.01$ trabajos.

In [2]:
# Registros
A = [] # Tiempo en la que llega una solicitud
D = [] # Tiempo en el cual se procesa una solicitud
R = [] # Intervalo (t0, t1) donde t0 es cuando comienza el reposo y t1 cuando termina
N = [0] # Cola de solicitudes
N_T = [0] # Tiempo en el cual se cambia la cola de solicitudes

def lamda_t(t: float):
    t_m = t % 10

    if t_m <= 5:
        return 3*t_m + 4
    else:
        return -3*t_m + 34

def t_nueva_solicitud():
    lamda_cota = 19
    t = 0

    while True:
        t -= math.log(1 - np.random.random()) / lamda_cota
        V = np.random.random()

        if V < lamda_t(t) / lamda_cota:
            return t

def t_servicio():
    # Exponencial con lambda = 25 (25 servicios por hora)
    return -math.log(1 - np.random.random()) / 25

def t_reposo():
    # Uniforme({0, 0.3})
    return np.random.random() * 0.3

def punto_a(T: int = 100):
    # Inicialización
    t, NA, ND = 0, 0, 0
    n = 0

    tA = t + t_nueva_solicitud()
    tD = math.inf

    # Como n = 0, se toma un descanso inmediatamente.
    tR = t + t_reposo()
    inicio_reposo = t

    while tA < math.inf or tD < math.inf:
        evento_prox = min(tA, tD, tR)

        # Llega nuevo solicitud
        if evento_prox == tA:
            t = tA
            NA += 1
            n += 1

            A.append(t)
            N.append(n)
            N_T.append(t)

            tA = t + t_nueva_solicitud()
            if tA > T:
                # El tiempo de la nueva solicitud supero T, por lo que no se
                # reciben nuevas solicitudes y solo se procesa las solicitudes
                # en la cola
                tA = math.inf

                print(
                    f"Ya pasaron las {T} horas, por lo que ya no llegan mas",
                    f"solicitudes. Quedan {n} clientes por atender.\n")

            if n == 1 and tR == math.inf:
                tD = t + t_servicio()
            elif n == 1 and tR < math.inf:
                tD = t + (tR - inicio_reposo) + t_servicio()

        elif evento_prox == tD:
            # Se termina de procesar una solicitud
            t = tD
            ND += 1
            n -= 1

            D.append(t)
            N.append(n)
            N_T.append(t)

            if n > 0:
                # Se añade el siguiente evento de atención del servidor
                tD = t + t_servicio()
            else:
                # Se vacio la cola, por lo que, el servidor entra en reposo
                tD = math.inf
                tR = t + t_reposo()
                inicio_reposo = t

        else:
            # Termina tiempo de reposo
            t = tR

            R.append((inicio_reposo, t))

            if n > 0:
                # Hay clientes esperando, termina el descanso y empieza a atender
                tR = math.inf
                tD = t + t_servicio()
            else:
                # Sigue vacío, se toma otro descanso inmediatamente
                tR = t + t_reposo()
                inicio_reposo = t

    print(f"Simulación finalizada a las {t:.2f} horas")
    print(f"Total de llegadas registradas: {len(A)}")
    print(f"Total de servicios completados: {len(D)}")
    print(f"Total de descansos tomados: {len(R)}")
    print()

    print("Tiempos de llegada (A):", A[:10], "...")
    print("Tiempos de salida (D):", D[:10], "...")
    print("Intervalos de descanso (R):", R[:5], "...")
    print("Evolución de la cola (N):", N[:10], "...")

punto_a()

Ya pasaron las 100 horas, por lo que ya no llegan mas solicitudes. Quedan 2 clientes por atender.

Simulación finalizada a las 100.10 horas
Total de llegadas registradas: 463
Total de servicios completados: 463
Total de descansos tomados: 544

Tiempos de llegada (A): [0.010281786741830347, 0.027294898255854523, 0.3153171224009245, 0.32428543649462815, 0.3710839456427857, 0.5021227712491046, 0.941932050330901, 1.0763695841236818, 1.247868659008517, 1.3205058171996424] ...
Tiempos de salida (D): [0.12290815841161443, 0.1758074345264793, 0.5706504846839947, 0.5835290077471771, 0.600949426737961, 0.6157317181741658, 1.1174523434174597, 1.1397723194162044, 1.4638302975606947, 1.4781143862766557] ...
Intervalos de descanso (R): [(0, 0.09755589959823238), (0.1758074345264793, 0.2669365146382954), (0.2669365146382954, 0.5489583000069982), (0.6157317181741658, 0.8022847484645844), (0.8022847484645844, 1.0907819588765633)] ...
Evolución de la cola (N): [0, 1, 2, 1, 0, 1, 2, 3, 4, 3] ...
